# conv-channel-sum — worked example 3: A 1x1 conv is a per-pixel matmul that sums over IC

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-channel-sum`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A 1x1 convolution has no spatial extent (`KH=KW=1`), so at every pixel it just takes the length-`IC` channel vector and maps it to a length-`OC` vector by summing `IC` weighted terms. That is exactly a matrix multiply `(OC, IC) @ (IC,)` applied independently at each spatial location — the clearest possible view of conv2d's in-channel sum.

## Worked solution

We show that the IC sum in a 1x1 conv is literally a matmul over the channel axis.

**Step 1 — squeeze the kernel.** `weight` is `(OC, IC, 1, 1)`. Dropping the trailing singleton spatial axes with `rearrange(weight, 'oc ic 1 1 -> oc ic')` leaves a plain `(OC, IC)` weight matrix.

**Step 2 — line up the channel axis for matmul.** The input `x` is `(B, IC, H, W)`. We want to contract IC against the kernel's second axis. Using `einsum('oi,bihw->bohw', W2d, x)` multiplies `W2d[o, i]` by `x[b, i, h, w]` and sums over `i` (IC). The repeated index `i` is the contracted in-channel axis; `o` (OC) replaces it in the output.

**Step 3 — why it matches conv2d.** For a 1x1 kernel there is no spatial summation (KH=KW=1), so the only sum a conv performs is over IC. The einsum performs that same sum at every `(h, w)` independently, leaving the spatial grid untouched — identical to `F.conv2d`.

**Step 4 — verify.** Comparing to `F.conv2d(x, weight)` confirms the equivalence to fp tolerance, making the IC contraction concrete as a per-pixel linear map.

In [ ]:
import torch.nn.functional as F

def pointwise_conv_as_matmul(x, weight):
    W2d = rearrange(weight, 'oc ic 1 1 -> oc ic')      # (OC, IC)
    y = t.einsum('oi,bihw->bohw', W2d, x)              # sum over i = IC
    return y

t.manual_seed(0)
x = t.randn(2, 6, 5, 5)
weight = t.randn(3, 6, 1, 1)
y = pointwise_conv_as_matmul(x, weight)
ref = F.conv2d(x, weight)
print(y.shape, t.allclose(y, ref, atol=1e-5))